# Train DeLPHI Pole Prediction Model

This notebook shows how to train the **GeoHierK3Transformer** pole prediction model using **DeLPHI** (`lc_pipeline`) with DAMIT lightcurve data.

## What This Notebook Does

1. **Install** dependencies (PyTorch, lc_pipeline)
2. **Download** DAMIT data (or use your own)
3. **Configure** training hyperparameters
4. **Train** the model with multi-epoch training
5. **Validate** and save the best checkpoint

## Training Details

- **Model**: GeoHierK3Transformer (d_model=128, 4 heads, 4 layers, ~994K params)
- **Loss**: oracle_softmin + continuous_diversity + batch_variance + contrastive
- **Dataset**: Multi-epoch (each observing epoch is a separate training sample)
- **Typical training time**: ~5 minutes per fold on GPU

## Expected Results

With 174 DAMIT asteroids (QF>=3), expect:
- **Oracle@K=3 mean**: ~19 deg (asteroid-level, 5-fold CV)
- **Oracle@K=3 pooled median**: ~17 deg
- **Convergence**: Usually within 6-10 epochs (early stopping)

---

## Step 1: Installation & Setup

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q numpy pandas tqdm scipy pydantic astropy

# Install lc_pipeline from GitHub
!pip install -q git+https://github.com/hangbin9/lc_dl.git

print("Installation complete")

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json
from pathlib import Path

from lc_pipeline.data.single_epoch_dataset import create_single_epoch_dataloaders
from lc_pipeline.models.geo_hier_k3_transformer import GeoHierK3Transformer
from lc_pipeline.training.losses_axisnet import combined_loss
from lc_pipeline.evaluation.eval_axisnet import evaluate_fold

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# Set random seeds for reproducibility
SEED = 777
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

## Step 2: Data Setup

You need three things to train:

1. **Lightcurve CSVs**: DAMIT format, 8 columns (time, brightness, sun_xyz, obs_xyz). One file per asteroid.
2. **Spin solutions**: JSON files with ground truth pole vectors. One file per asteroid.
3. **Period JSON**: Maps asteroid IDs to rotation periods in hours.

Download DAMIT data from: https://astro.troja.mff.cuni.cz/projects/damit/

Place the data in this structure:
```
data/
├── damit_csv_qf_ge_3/          # Lightcurve CSVs
│   ├── asteroid_101.csv
│   └── ...
├── damit_spins_complete/        # Spin solution JSONs
│   ├── asteroid_101.json
│   └── ...
└── periods.json                 # {"asteroid_101": 8.34, ...}
```

In [ ]:
# For Google Colab: upload data or mount Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Set data paths (adjust to match your setup)
CSV_DIR = "data/damit_csv_qf_ge_3"
SPIN_ROOT = "data/damit_spins_complete"
PERIOD_JSON = "data/periods.json"
OUTPUT_DIR = "checkpoints_output"

# Verify data exists
for path, name in [(CSV_DIR, "CSV dir"), (SPIN_ROOT, "Spin root"), (PERIOD_JSON, "Period JSON")]:
    if Path(path).exists():
        print(f"  {name}: {path} (found)")
    else:
        print(f"  {name}: {path} (NOT FOUND - please update path)")

## Step 3: Configuration

Training hyperparameters (production defaults).

These are the same settings used to produce the shipped checkpoints.

In [ ]:
# Training configuration (production defaults)
FOLD = 0              # Which fold to train (0-4)
EPOCHS = 200          # Max epochs (early stopping usually kicks in around 6-10)
PATIENCE = 50         # Early stopping patience
BATCH_SIZE = 32       # Batch size
LEARNING_RATE = 3e-4  # AdamW learning rate
WEIGHT_DECAY = 1e-4   # AdamW weight decay

# Architecture (do not change for production model)
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1

# Loss hyperparameters (production values)
SOFTMIN_TAU = 5.0     # Oracle softmin temperature (degrees)
LAMBDA_DIV = 0.5      # Continuous diversity weight
DIV_SIGMA = 15.0      # Diversity sigma (degrees)
LAMBDA_VAR = 5.0      # Batch variance weight
LAMBDA_SIM = 2.0      # Contrastive similarity weight

print("Training Configuration:")
print(f"  Fold: {FOLD}")
print(f"  Epochs: {EPOCHS} (with patience={PATIENCE})")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Model: d_model={D_MODEL}, n_heads={N_HEADS}, n_layers={N_LAYERS}")

## Step 4: Create Data Splits and Dataloaders

Using the production multi-epoch dataset where each observing epoch is a separate training sample.

In [ ]:
# Load object IDs from CSV directory
csv_files = sorted(Path(CSV_DIR).glob("*.csv"))
object_ids = [f.stem for f in csv_files]
print(f"Found {len(object_ids)} asteroids")

# Create deterministic fold split
np.random.seed(1337)
shuffled = np.random.permutation(object_ids)
fold_size = len(shuffled) // 5

val_start = FOLD * fold_size
val_end = (FOLD + 1) * fold_size
val_ids = list(shuffled[val_start:val_end])
train_ids = list(shuffled[:val_start]) + list(shuffled[val_end:])

print(f"Fold {FOLD}: {len(train_ids)} train, {len(val_ids)} val")

# Create dataloaders (multi-epoch mode)
train_loader, val_loader = create_single_epoch_dataloaders(
    csv_dir=CSV_DIR,
    spin_root=SPIN_ROOT,
    train_ids=train_ids,
    val_ids=val_ids,
    batch_size=BATCH_SIZE,
    min_gap_days=30.0,
    use_geometry=False,
    period_json=PERIOD_JSON,
)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## Step 5: Initialize Model

In [ ]:
# Create the production model
model = GeoHierK3Transformer(
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers_window=N_LAYERS,
    n_layers_cross=0,          # No cross-window encoder (removed in production)
    n_feature_input=13,        # 3 temporal + 6 geometry zeros + 4 period
    include_quality_head=False, # No quality head in production
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

# Optimizer and scheduler
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

## Step 6: Training Loop

Train with early stopping on validation oracle error.

In [ ]:
import time

output_dir = Path(OUTPUT_DIR) / f"fold_{FOLD}"
output_dir.mkdir(parents=True, exist_ok=True)

best_oracle = float('inf')
best_epoch = -1
patience_counter = 0
history = []

print(f"Training fold {FOLD}...")
print(f"Output: {output_dir}")
print()

t_start = time.time()

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for batch in train_loader:
        tokens = batch['tokens'].to(device)
        mask = batch['mask'].to(device)
        solutions_list = [
            s.to(device) if s is not None else None
            for s in batch['solutions']
        ]

        optimizer.zero_grad()
        poles, quality_logits = model(tokens, mask)

        loss_dict = combined_loss(
            poles, quality_logits, solutions_list,
            lambda_div=LAMBDA_DIV,
            lambda_q=0.0,
            lambda_var=LAMBDA_VAR,
            lambda_sim=LAMBDA_SIM,
            softmin_tau_deg=SOFTMIN_TAU,
            div_sigma_deg=DIV_SIGMA,
        )
        loss = loss_dict['loss']
        loss_val = loss.item()

        if not (np.isnan(loss_val) or np.isinf(loss_val)):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss_val

        n_batches += 1

    avg_loss = epoch_loss / max(n_batches, 1)
    scheduler.step()

    # --- Validate ---
    model.eval()
    fold_results = evaluate_fold(model, val_loader, device=device)
    oracle_med = fold_results['metrics']['oracle_median_deg']
    if oracle_med is None:
        oracle_med = float('inf')

    history.append({
        'epoch': epoch,
        'train_loss': avg_loss,
        'oracle_median_deg': float(oracle_med),
    })

    # Log
    elapsed = time.time() - t_start
    print(
        f"Epoch {epoch:3d} | Loss: {avg_loss:.4f} | "
        f"Val Oracle: {oracle_med:.2f} deg | "
        f"Time: {elapsed:.0f}s"
    )

    # Best model tracking
    if oracle_med < best_oracle:
        best_oracle = oracle_med
        best_epoch = epoch
        patience_counter = 0
        print(f"  -> New best: {oracle_med:.2f} deg (saving)")
        torch.save({
            'model_state_dict': model.state_dict(),
            'fold_idx': FOLD,
            'epoch': epoch,
            'best_oracle_median': best_oracle,
        }, output_dir / "checkpoint_best_oracle.pt")
    else:
        patience_counter += 1

    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

total_time = time.time() - t_start
print(f"\nTraining complete in {total_time:.0f}s")
print(f"Best oracle: {best_oracle:.2f} deg at epoch {best_epoch}")

# Save history
with open(output_dir / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

## Step 7: Visualize Training

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = [h['epoch'] for h in history]

# Loss curve
ax1.plot(epochs_x, [h['train_loss'] for h in history])
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Training Loss")
ax1.grid(True, alpha=0.3)

# Oracle error curve
ax2.plot(epochs_x, [h['oracle_median_deg'] for h in history])
ax2.axhline(best_oracle, color='red', linestyle='--', label=f'Best: {best_oracle:.2f} deg')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Oracle Median Error (degrees)")
ax2.set_title("Validation Oracle@K=3")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Test the Trained Model

In [ ]:
# Load best checkpoint and run inference
from lc_pipeline import analyze

checkpoint_path = output_dir / "checkpoint_best_oracle.pt"
print(f"Checkpoint: {checkpoint_path}")
print(f"Best oracle: {best_oracle:.2f} deg at epoch {best_epoch}")
print()

# Test with example data (if available)
example_path = Path("examples/asteroid_101.csv")
if example_path.exists():
    import pandas as pd
    data = pd.read_csv(example_path, comment='#').dropna().values
    result = analyze([data], "asteroid_101", fold=FOLD)
    print(f"Period: {result.period.period_hours:.2f} h")
    print(f"Best pole: lambda={result.best_pole.lambda_deg:.1f}, beta={result.best_pole.beta_deg:.1f}")
    print(f"Total candidates: {len(result.poles)}")
else:
    print("No example data found. Use run_pole_prediction.py to test your model.")

## Summary

Training complete. Your checkpoint is saved and ready for inference.

**Command-line alternative**: For a simpler workflow, use:
```bash
python train_pole_model.py \
    --csv-dir data/damit_csv_qf_ge_3 \
    --spin-root data/damit_spins_complete \
    --period-json data/periods.json \
    --outdir checkpoints_new \
    --folds 0,1,2,3,4
```

See `docs/USER_GUIDE.md` for detailed documentation.